# Evaluation | Generating Ground Truth Data

To evaluate search, we need a dataset of queries where we know which document is the correct answer. This is called ground truth (or gold standard) data.

For each query in our ground truth dataset, we know which document in the knowledge base is relevant. When we run a search, we check whether the results include the correct document.

There are several ways to get ground truth data:

Human annotators look at documents and write queries (best quality, expensive)
Collect real user queries and label them (requires a running system)
Generate synthetic data with an LLM (what we'll do)
We don't have a production system yet, so we'll use an LLM to generate questions. For each FAQ document, we ask the LLM to create 5 questions that this document would answer. Then we know that for each generated question, the source document is the correct answer.




In [5]:
from ingest import load_faq_data
documents = load_faq_data()

In [6]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [7]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [8]:
documents_llm[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [9]:
documents = documents_llm

## Generating questions with structured output
We use an LLM to generate questions for each document.

This is the first time we're using structured output in the course.
With structured output, we ask the LLM to return data in a specific
format instead of free-form text. For example, instead of getting a
paragraph that contains questions, we can ask for a Python object with
a `questions` field.

This is useful when code will process the output. The model returns the
same structure every time. We can access the generated questions
directly instead of parsing text manually.

We want the output as a list of strings, so we define that structure
with a Pydantic model:

In [10]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]


The instructions for the LLM:


In [11]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

We ask the LLM to use different wording from the original document.
This makes the evaluation more realistic - real users won't phrase
their questions the same way as the FAQ.

Call the LLM for one document:


In [12]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [13]:
doc

{'id': 'ab183bd688',
 'course': 'machine-learning-zoomcamp',
 'section': 'Miscellaneous',
 'question': "My homework answer doesn't match any of the options",
 'answer': "Common causes, in order of frequency:\n\n1. Wrong column slice or filter — apply filters BEFORE selecting columns / `.head(n)` / `.values`.\n2. Log transform applied where it shouldn't be (or not applied where it should).\n3. Rounding too early — only round the final answer, not intermediate values, unless explicitly told to.\n4. Different sklearn / numpy / Python versions — pin them via `requirements.txt`, `Pipfile.lock`, or `uv.lock`.\n5. Different train/val/test split logic — `train_test_split` shuffles by default; manual `np.random.shuffle` produces a different ordering than sklearn's.\n\nIf after these checks your answer still doesn't match, pick the closest option — the homework explicitly allows it."}

In [14]:
import json
user_prompt = json.dumps(doc)
user_prompt

'{"id": "ab183bd688", "course": "machine-learning-zoomcamp", "section": "Miscellaneous", "question": "My homework answer doesn\'t match any of the options", "answer": "Common causes, in order of frequency:\\n\\n1. Wrong column slice or filter \\u2014 apply filters BEFORE selecting columns / `.head(n)` / `.values`.\\n2. Log transform applied where it shouldn\'t be (or not applied where it should).\\n3. Rounding too early \\u2014 only round the final answer, not intermediate values, unless explicitly told to.\\n4. Different sklearn / numpy / Python versions \\u2014 pin them via `requirements.txt`, `Pipfile.lock`, or `uv.lock`.\\n5. Different train/val/test split logic \\u2014 `train_test_split` shuffles by default; manual `np.random.shuffle` produces a different ordering than sklearn\'s.\\n\\nIf after these checks your answer still doesn\'t match, pick the closest option \\u2014 the homework explicitly allows it."}'

Create the messages:


In [15]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

Until now we called `responses.create` and read `response.output_text`.
For structured output we switch to `responses.parse` and pass
`text_format=Questions`, which tells the API to return our class instead
of free text.

Call the model:

In [16]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

The parsed object is available in `response.output_parsed`:

In [17]:
response.output_parsed.questions

['Why does my homework result not match any of the multiple-choice answers, and what should I check first?',
 'Could a wrong column slice or applying .head() too early make my ML homework answer off from the options?',
 'How do log transforms, rounding, or different sklearn/numpy versions cause answers to differ from the expected choice?',
 'Why does my train/validation/test split look different from the solution even though my code seems correct?',
 'If I’ve already checked the usual issues and my answer still isn’t exact, should I choose the closest option in the homework?']

In [18]:
doc

{'id': 'ab183bd688',
 'course': 'machine-learning-zoomcamp',
 'section': 'Miscellaneous',
 'question': "My homework answer doesn't match any of the options",
 'answer': "Common causes, in order of frequency:\n\n1. Wrong column slice or filter — apply filters BEFORE selecting columns / `.head(n)` / `.values`.\n2. Log transform applied where it shouldn't be (or not applied where it should).\n3. Rounding too early — only round the final answer, not intermediate values, unless explicitly told to.\n4. Different sklearn / numpy / Python versions — pin them via `requirements.txt`, `Pipfile.lock`, or `uv.lock`.\n5. Different train/val/test split logic — `train_test_split` shuffles by default; manual `np.random.shuffle` produces a different ordering than sklearn's.\n\nIf after these checks your answer still doesn't match, pick the closest option — the homework explicitly allows it."}

In [19]:
result = response.output_parsed

print(result)

questions=['Why does my homework result not match any of the multiple-choice answers, and what should I check first?', 'Could a wrong column slice or applying .head() too early make my ML homework answer off from the options?', 'How do log transforms, rounding, or different sklearn/numpy versions cause answers to differ from the expected choice?', 'Why does my train/validation/test split look different from the solution even though my code seems correct?', 'If I’ve already checked the usual issues and my answer still isn’t exact, should I choose the closest option in the homework?']


In [20]:
documents.__len__()

113

## Reusable utilities

We'll need this pattern again in other evaluation sections today, so
we put it in a reusable helper.

It contains helper functions we'll reuse in this module:

- `llm_structured`: calls the OpenAI API with structured output
- `llm_structured_retry`: retries structured-output calls when a
  request fails
- `calc_price`: calculates the price from token usage
- `calc_total_price`: calculates the total price from multiple usage
  objects
- `map_progress`: runs work in parallel and tracks progress. We'll use it
  in the next lesson.

Import the structured-output helper:

In [21]:
from evaluation_utils import llm_structured

In [22]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

["Why doesn't my homework result match any of the multiple-choice answers, even after I checked my code?", 'What are the most common reasons a machine learning homework answer ends up different from the options given?', 'Could I be slicing the dataframe wrong if my final number is not in the list of choices?', 'Does rounding too early or using a log transform in the wrong place change the homework answer a lot?', "If my computed answer still doesn't match any option after checking everything, should I just choose the nearest one?"]


In [23]:
usage

ResponseUsage(input_tokens=352, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=115, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=467)

## Tracking cost

The response also contains token usage:

```python
usage.input_tokens, usage.output_tokens
```

As in the agents module, we calculate the price from `response.usage`.

Import the price helper:

In [24]:
from evaluation_utils import calc_price

In [25]:
cost = calc_price(usage)

cost

{'input_cost': 0.000264,
 'output_cost': 0.0005175000000000001,
 'total_cost': 0.0007815000000000001}

Now convert these questions into ground truth records:


In [26]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': "Why doesn't my homework result match any of the multiple-choice answers, even after I checked my code?",
  'document': 'ab183bd688'},
 {'question': 'What are the most common reasons a machine learning homework answer ends up different from the options given?',
  'document': 'ab183bd688'},
 {'question': 'Could I be slicing the dataframe wrong if my final number is not in the list of choices?',
  'document': 'ab183bd688'},
 {'question': 'Does rounding too early or using a log transform in the wrong place change the homework answer a lot?',
  'document': 'ab183bd688'},
 {'question': "If my computed answer still doesn't match any option after checking everything, should I just choose the nearest one?",
  'document': 'ab183bd688'}]

Each record has two fields:

- `question`: the question generated by the LLM
- `document`: the ID of the FAQ document that should answer the question

The `document` field connects the generated question to the document
that contains the answer. Later, when we evaluate search, we'll ask the
search engine the generated question. Then we'll check if it retrieves
the document with this ID.

We now know how to generate and store questions for one document. In
the next lesson, we'll run this for all LLM Zoomcamp FAQ documents and
save the full ground truth dataset.


# Generating Ground Truth for All Documents
We want to do the same thing for every document in the FAQ dataset.
For each document, we generate questions and save them as ground truth
records.

For this part, we'll use `tqdm` for progress bars and `pandas` for
saving the final CSV.


In [27]:
import pandas as pd

In [28]:
pd.DataFrame(records)

,question,document
0,Why doesn't my homework result match any of th...,ab183bd688
1,What are the most common reasons a machine lea...,ab183bd688
2,Could I be slicing the dataframe wrong if my f...,ab183bd688
3,Does rounding too early or using a log transfo...,ab183bd688
4,If my computed answer still doesn't match any ...,ab183bd688


The processing function takes one document and turns it into ground
truth records.

For each document, we:

- convert the document to JSON so we can send it to the LLM
- ask the LLM to return a `Questions` object
- create one ground truth record for each generated question

Each record contains the generated question and the ID of the document
that should answer the question.

When we send many requests, one of them might fail. We don't want the
entire batch to fail because of one temporary error.

Import the retry helper from `evaluation_utils.py`:

In [29]:
from evaluation_utils import llm_structured_retry

`llm_structured` makes one structured-output call. `llm_structured_retry`
wraps the same call in a retry loop. If one request fails because of a
temporary API or network issue, it waits briefly and tries again.

Use it in the processing function:

In [30]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [31]:
generate_ground_truth(doc)

([{'question': 'Why does my homework result not match any of the multiple-choice options, even after I followed the notebook?',
   'document': 'ab183bd688'},
  {'question': 'What are the most common reasons an ML Zoomcamp homework answer comes out different from the listed choices?',
   'document': 'ab183bd688'},
  {'question': 'Could using the wrong slice or filtering after `.head()` / `.values` be the reason my answer is off?',
   'document': 'ab183bd688'},
  {'question': 'Why would my solution differ if I rounded numbers too early or applied a log transform in the wrong place?',
   'document': 'ab183bd688'},
  {'question': "If my result still doesn't match any option after checking splits and library versions, should I just choose the closest answer?",
   'document': 'ab183bd688'}],
 ResponseUsage(input_tokens=352, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=124, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_to

Try it for the first 5 documents.

Import `tqdm` and run the loop:

In [32]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

## Parallel processing

Running the calls one after another wastes most of the time waiting on
the network. Each request just sits there until OpenAI responds, so we
can fire several at once and wait on them together. We process the
documents in parallel and track progress while the requests run.

One caution: don't open too many connections at once, or you'll hit the
provider's rate limits. Five or six workers is a safe default here.

Import `ThreadPoolExecutor`:


In [33]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

This submits one job per document, updates the progress bar when a job
finishes, and collects the results. If you want a more detailed
explanation of `ThreadPoolExecutor` and futures, ask ChatGPT to walk
through this helper line by line.

Then replace the loop with the parallel version:

In [34]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [35]:
results[1]

([{'question': 'I signed up for the LLM Zoomcamp — when should I get the confirmation email?',
   'document': '977bf7786c'},
  {'question': 'Do I actually need to wait for a registration confirmation before starting the course?',
   'document': '977bf7786c'},
  {'question': 'If I didn’t register, can I still begin the LLM Zoomcamp and submit homework while it’s open?',
   'document': '977bf7786c'},
  {'question': 'Is the course checking homework submissions against a list of registered students?',
   'document': '977bf7786c'},
  {'question': 'What’s the point of registering for the LLM Zoomcamp if it doesn’t affect access?',
   'document': '977bf7786c'}],
 ResponseUsage(input_tokens=238, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=104, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=342))

`generate_ground_truth` returns two things for each document: the
generated records and the token usage.

Split those into separate lists:

In [36]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [37]:
ground_truth[2]

{'question': 'If I start the course after it’s been running for a while, can I still get involved?',
 'document': '74eb249bbf'}

With 5 questions per document, you should get roughly 5x the number of
documents.


Calculate the total cost:


In [38]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08812950000000004

We'll calculate total cost several times in this module, so the utility
file has a helper for it:

In [39]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08812950000000004

Create a dataframe so we can look at the records as a table and save
them as a CSV file.

Create the dataframe:

In [40]:
df_ground_truth = pd.DataFrame(ground_truth)

Because we generated the questions from specific documents, we know
which document is correct for each question. We now have the ground
truth we need for evaluation.

Save it for later use:

In [41]:
df_ground_truth.to_csv("../data/ground_truth-new.csv", index=False)

We generated this file for the course materials on May 29, 2026. The
run used 79 LLM Zoomcamp documents and produced 395 questions.

The FAQ data can change over time. If you run the notebook later, you
may see different documents and generated questions. Token usage, cost,
and search evaluation results may also change.

The total cost was $0.057187, about 6 cents.

In [42]:
df_ground_truth.__len__()

565

If you don't want to generate the questions yourself, download the file
we prepared:

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

wget -O data/ground_truth-new.csv ${PREFIX}/04-evaluation/data/ground_truth-new.csv
```